# Fill in Missing ROIs

Enriches `extra_movies_raw.csv` with financial data from a Kaggle TMDB archive,
then produces clean per-file CSVs and a combined master dataset with inflation-adjusted ROI.

**Pipeline stages:**
1. Unzip the Kaggle TMDB archive from Google Drive
2. Load both the project dataset and the Kaggle dataset
3. Merge Kaggle budget/gross into `extra_movies_raw` (TMDB ID first, IMDB ID fallback)
4. Filter rows to those with valid financials and compute ROI for both CSVs
5. Check data completeness (ROI + poster + script)
6. Concatenate valid rows into `master_movie_data.csv`
7. Back-fill missing inflation-adjusted values using the `cpi` package

## Imports & Constants

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from bs4 import BeautifulSoup

BASE_PATH       = '/content/gdrive/Shareddrives/FML_FINAL/Data/'
FILE_PATH_EXTRA = BASE_PATH + 'extra_movies_raw.csv'
FILE_PATH_MAIN  = BASE_PATH + 'movies_raw.csv'
MASTER_PATH     = BASE_PATH + 'master_movie_data.csv'
ZIP_PATH        = BASE_PATH + 'archive.zip'

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 2. Unzip Kaggle Archive

Expects `archive.zip` (TMDB movies metadata) at `BASE_PATH`.
After unzipping, `movies_metadata.csv` should appear in `/content/kaggle_data/`.

In [ ]:
!unzip -o "{ZIP_PATH}" -d /content/kaggle_data
!ls /content/kaggle_data

## 3. Load Datasets

In [ ]:
df        = pd.read_csv(FILE_PATH_EXTRA)
kaggle_df = pd.read_csv('/content/kaggle_data/movies_metadata.csv', low_memory=False)

print(f"extra_movies_raw : {len(df):,} rows")
print(f"Kaggle dataset   : {len(kaggle_df):,} rows")

## 4. Clean Kaggle Financials

The Kaggle dataset uses `0` as a null sentinel for budget and revenue,
and some `id` values are non-numeric (bad rows). Both are handled here.

In [ ]:
# Drop rows with non-numeric IDs
kaggle_df = kaggle_df[pd.to_numeric(kaggle_df['id'], errors='coerce').notna()].copy()
kaggle_df['tmdb_id']   = kaggle_df['id'].astype(int)
kaggle_df['budget_kg'] = pd.to_numeric(kaggle_df['budget'],  errors='coerce')
kaggle_df['gross_kg']  = pd.to_numeric(kaggle_df['revenue'], errors='coerce')

# Treat 0 as missing
kaggle_df.loc[kaggle_df['budget_kg'] == 0, 'budget_kg'] = None
kaggle_df.loc[kaggle_df['gross_kg']  == 0, 'gross_kg']  = None

kaggle_df['imdb_id_kg'] = kaggle_df['imdb_id'].astype(str).str.strip()

kaggle_clean = (
    kaggle_df[['tmdb_id', 'imdb_id_kg', 'budget_kg', 'gross_kg']]
    .drop_duplicates('tmdb_id')
)

print(f"Kaggle rows with budget : {kaggle_clean['budget_kg'].notna().sum():,}")
print(f"Kaggle rows with gross  : {kaggle_clean['gross_kg'].notna().sum():,}")

## 5. Merge Kaggle Financials into `extra_movies_raw`

Two-pass merge: TMDB ID first, then IMDB ID as a fallback for rows still missing
budget or gross after the first pass. Existing non-zero values are never overwritten.

In [ ]:
df['tmdb_id'] = pd.to_numeric(df['tmdb_id'], errors='coerce')
df['imdb_id']  = df['imdb_id'].astype(str).str.strip()

before_budget = df['budget_raw'].notna().sum()
before_gross  = df['gross_raw'].notna().sum()

# ── Pass 1: merge on tmdb_id ──────────────────────────────────────────────
df = df.merge(kaggle_clean[['tmdb_id', 'budget_kg', 'gross_kg']], on='tmdb_id', how='left')
df['budget_raw'] = df['budget_raw'].where(df['budget_raw'].notna() & (df['budget_raw'] != 0), df['budget_kg'])
df['gross_raw']  = df['gross_raw'].where(df['gross_raw'].notna() & (df['gross_raw'] != 0),  df['gross_kg'])
df.drop(columns=['budget_kg', 'gross_kg'], inplace=True)

after_tmdb_budget = df['budget_raw'].notna().sum()
after_tmdb_gross  = df['gross_raw'].notna().sum()
print(f"After tmdb_id merge  — budget filled: {after_tmdb_budget - before_budget}  "
      f"gross filled: {after_tmdb_gross - before_gross}")

# ── Pass 2: imdb_id fallback for rows still missing ───────────────────────
still_missing = df['budget_raw'].isna() | df['gross_raw'].isna()
print(f"Still missing after tmdb merge: {still_missing.sum()} rows — trying imdb_id fallback...")

kaggle_imdb = (
    kaggle_clean[['imdb_id_kg', 'budget_kg', 'gross_kg']]
    .rename(columns={'imdb_id_kg': 'imdb_id'})
)
df = df.merge(kaggle_imdb, on='imdb_id', how='left')
df['budget_raw'] = df['budget_raw'].where(df['budget_raw'].notna() & (df['budget_raw'] != 0), df['budget_kg'])
df['gross_raw']  = df['gross_raw'].where(df['gross_raw'].notna() & (df['gross_raw'] != 0),  df['gross_kg'])
df.drop(columns=['budget_kg', 'gross_kg'], inplace=True)

after_imdb_budget = df['budget_raw'].notna().sum()
after_imdb_gross  = df['gross_raw'].notna().sum()
print(f"After imdb_id fallback — budget filled: {after_imdb_budget - after_tmdb_budget}  "
      f"gross filled: {after_imdb_gross - after_tmdb_gross}")

# Save merged (unfiltered) so the next step can reload both files uniformly
df.to_csv(FILE_PATH_EXTRA, index=False)
print(f"Saved extra_movies_raw.csv ({len(df):,} rows)")

## 6. Filter & Compute ROI for Both Files

Rows without both a positive `budget_raw` and `gross_raw` are dropped from each file.
Both raw ROI (`roi`, `log_roi`) and inflation-adjusted ROI (`roi_2025`, `log_roi_2025`)
are computed where the required columns are present. Each file is saved in place.

In [ ]:
for file_path in [FILE_PATH_MAIN, FILE_PATH_EXTRA]:
    file_name = file_path.split('/')[-1]
    print(f"\n--- {file_name} ---")
    try:
        df_cur    = pd.read_csv(file_path)
        n_before  = len(df_cur)

        # Drop rows missing valid financials
        df_cur = df_cur[
            df_cur['budget_raw'].notna() & df_cur['gross_raw'].notna() &
            (df_cur['budget_raw'] > 0)   & (df_cur['gross_raw'] > 0)
        ].copy()
        print(f"  Dropped {n_before - len(df_cur)} rows missing financials. Remaining: {len(df_cur):,}")

        # Raw ROI
        df_cur['roi']     = (df_cur['gross_raw'] - df_cur['budget_raw']) / df_cur['budget_raw']
        df_cur['log_roi'] = np.log1p(df_cur['roi'])

        # Inflation-adjusted ROI (only when CPI-adjusted columns exist)
        if {'budget_2025', 'gross_2025'}.issubset(df_cur.columns):
            adj_mask = (
                df_cur['budget_2025'].notna() & df_cur['gross_2025'].notna() &
                (df_cur['budget_2025'] > 0)   & (df_cur['gross_2025'] > 0)
            )
            df_cur.loc[adj_mask, 'roi_2025'] = (
                (df_cur.loc[adj_mask, 'gross_2025'] - df_cur.loc[adj_mask, 'budget_2025']) /
                 df_cur.loc[adj_mask, 'budget_2025']
            )
            df_cur.loc[adj_mask, 'log_roi_2025'] = np.log1p(df_cur.loc[adj_mask, 'roi_2025'])
            print(f"  Computed 2025-adjusted ROI for {adj_mask.sum():,} rows.")

        df_cur.to_csv(file_path, index=False)
        print(f"  Saved {file_name}")

    except FileNotFoundError:
        print(f"  Error: {file_name} not found at {file_path}")

## 7. Check Data Completeness

Counts rows in each file that have all four fields required to reach
`master_movie_data.csv`: `roi`, `log_roi`, `poster_file`, and `script_file`.

In [ ]:
REQUIRED_COLS = ['roi', 'log_roi', 'poster_file', 'script_file']
total_valid   = 0

for file_path in [FILE_PATH_MAIN, FILE_PATH_EXTRA]:
    file_name = file_path.split('/')[-1]
    try:
        df_temp = pd.read_csv(file_path)
        if not all(c in df_temp.columns for c in REQUIRED_COLS):
            missing = [c for c in REQUIRED_COLS if c not in df_temp.columns]
            print(f"{file_name}: missing columns {missing}")
            continue
        has_roi     = df_temp['roi'].notna()
        has_log_roi = df_temp['log_roi'].notna()
        has_poster  = df_temp['poster_file'].notna() & (df_temp['poster_file'].astype(str).str.strip() != '')
        has_script  = df_temp['script_file'].notna() & (df_temp['script_file'].astype(str).str.strip() != '')
        valid_count  = (has_roi & has_log_roi & has_poster & has_script).sum()
        total_valid += valid_count
        print(f"{file_name}: {valid_count:,} complete rows")
    except Exception as e:
        print(f"{file_name}: error — {e}")

print(f"\nTotal across both files: {total_valid:,}")

## 8. Create Master Dataset

Keeps only rows with valid `roi`, `log_roi`, `poster_file`, and `script_file`
from each file, then concatenates them into `master_movie_data.csv`.

In [ ]:
def filter_complete_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Keeps rows with non-null, non-empty roi, log_roi, poster_file, and script_file."""
    has_roi     = df['roi'].notna()
    has_log_roi = df['log_roi'].notna()
    has_poster  = df['poster_file'].notna() & (df['poster_file'].astype(str).str.strip() != '')
    has_script  = df['script_file'].notna() & (df['script_file'].astype(str).str.strip() != '')
    return df[has_roi & has_log_roi & has_poster & has_script].copy()


df1 = filter_complete_rows(pd.read_csv(FILE_PATH_MAIN))
df2 = filter_complete_rows(pd.read_csv(FILE_PATH_EXTRA))

master_df = pd.concat([df1, df2], ignore_index=True)
master_df.to_csv(MASTER_PATH, index=False)

print(f"movies_raw.csv contribution:       {len(df1):,} rows")
print(f"extra_movies_raw.csv contribution: {len(df2):,} rows")
print(f"Master dataset total:              {len(master_df):,} rows")
print(f"Saved to: {MASTER_PATH}")
display(master_df.head())

## 9. Compute Missing Inflation Adjustments

Uses the `cpi` package (BLS CPI data) to fill `budget_2025`, `gross_2025`,
`roi_2025`, and `log_roi_2025` for rows that still lack adjusted values.
Inflation factors are pre-computed per unique year for efficiency.

In [ ]:
!pip install cpi

In [ ]:
import cpi

try:
    cpi.update()
    print(f"CPI data updated. Latest year available: {cpi.LATEST_YEAR}")
except Exception as e:
    print(f"Warning: could not update CPI data: {e}")

master_df = pd.read_csv(MASTER_PATH)
master_df['release_year'] = pd.to_numeric(master_df['release_year'], errors='coerce')

# Rows that have raw financials but no 2025-adjusted values yet
needs_adj = (
    (master_df['budget_2025'].isna() | (master_df['budget_2025'] == 0)) &
    master_df['budget_raw'].notna() & (master_df['budget_raw'] > 0) &
    master_df['release_year'].notna()
)
print(f"Rows needing inflation adjustment: {needs_adj.sum():,}")

# Pre-compute one inflation factor per unique year (avoid repeated API calls)
unique_years = [int(y) for y in master_df.loc[needs_adj, 'release_year'].dropna().unique()]
inflation_factors = {}
for year in unique_years:
    try:
        inflation_factors[year] = cpi.inflate(1.0, year) if 1913 <= year <= cpi.LATEST_YEAR else 1.0
    except Exception:
        inflation_factors[year] = 1.0
print(f"Inflation factors computed for {len(inflation_factors)} unique years.")


def _apply_inflation(row):
    factor     = inflation_factors.get(int(row['release_year']), 1.0)
    budget_adj = row['budget_raw'] * factor if pd.notna(row['budget_raw']) else np.nan
    gross_adj  = row['gross_raw']  * factor if pd.notna(row['gross_raw'])  else np.nan
    return pd.Series({
        'budget_2025':    budget_adj,
        'gross_2025':     gross_adj,
        'cpi_budget_year': int(row['release_year']) if pd.notna(budget_adj) else np.nan,
        'cpi_gross_year':  int(row['release_year']) if pd.notna(gross_adj)  else np.nan,
    })


adjusted = master_df[needs_adj].apply(_apply_inflation, axis=1)
master_df.loc[needs_adj, ['budget_2025', 'gross_2025', 'cpi_budget_year', 'cpi_gross_year']] = adjusted.values
print(f"Applied inflation adjustment to {needs_adj.sum():,} rows.")

# Recompute 2025-adjusted ROI for rows that now have both adjusted values
valid_adj = (
    master_df['budget_2025'].notna() & master_df['gross_2025'].notna() &
    (master_df['budget_2025'] > 0)
)
master_df.loc[valid_adj, 'roi_2025'] = (
    (master_df.loc[valid_adj, 'gross_2025'] - master_df.loc[valid_adj, 'budget_2025']) /
     master_df.loc[valid_adj, 'budget_2025']
)
master_df.loc[valid_adj, 'log_roi_2025'] = np.log1p(master_df.loc[valid_adj, 'roi_2025'])

master_df.to_csv(MASTER_PATH, index=False)
print(f"Saved updated master dataset to {MASTER_PATH}")

print("\nSample of adjusted rows:")
display(master_df[needs_adj][[
    'title', 'release_year', 'budget_raw', 'budget_2025', 'gross_raw', 'gross_2025', 'roi_2025'
]].head(10))